## **RouteOptimize: Optimizing Delivaries**

**1. Import Library & Module**

In [ ]:
# Import libraries
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

# Import src.distance_matrix
from src.distance_matrix import *


**2. Check feasibility for total demand and total supply:**

In [ ]:
# Retailers with weight > 1000kg
df[df['Esti_Demand_Per_Day_kg'] > 1000]

# Total demand across all retailers
total_demand = df['Esti_Demand_Per_Day_kg'].sum()
print('Total Demand by Past Data: ', total_demand)

# Total supply: 87 vehicles, Appox. 6500 Kgs per vehicle
total_supply = 87*6500
print('Total Supply Capacity: ', total_supply)


def check():
    if total_supply >= total_demand:
        print("Demand is fulfilling by the vehicles")
    else:
        print("Not fulfilling the total demand")

# Print feasibility Result
check()

Total Demand by Past Data:  538574
Total Supply Capacity:  565500
Demand is fulfilling by the vehicles


**3. Distance Matrix: Haversine Formula**

In [ ]:
# Get distance_matrix from src
distance_matrix

# Shape of the distance matrix
print("Distance matrix shape:", np.shape(distance_matrix))

# Check for any NaN values
print("NaN in matrix:", np.isnan(distance_matrix).sum())

Distance matrix shape: (1033, 1033)
NaN in matrix: 0


**4. JCCI logistic CVRP: Through OR-Tools**

In [ ]:

# Available vehicles: For Homogeneous fleet of Vehicles
vehicle_count = 87

# Load capacity of vehicle
vehicle_capacity = 6500

# Service time required at each customer location (in minutes)
SERVICE_TIME_MIN = 8

# Create the routing index manager.
# Translates internal solver indices to actual Node indices and vice versa
manager = pywrapcp.RoutingIndexManager(len(distance_matrix), vehicle_count, 0)

# Create the routing model.
# Object where constraints and objectives are added
routing = pywrapcp.RoutingModel(manager)

In [ ]:
# Define a callback_function to provide the travel distance between any two nodes
# Function used by the OR-Tools solver to calculate route costs

def distance_callback(i, j):
    # Convert solver internal indices to actual node indices.
    from_node, to_node = manager.IndexToNode(i), manager.IndexToNode(j)
    # Return the distance from the precomputed distance_matrix.
    return int(distance_matrix[from_node][to_node])


# This creates a transit_callback_index that can be used to set arc costs
transit_callback_index = routing.RegisterTransitCallback(distance_callback)

# Set the cost evaluator for all vehicles.
# Cost of traversing an arc in the graph, determined by the distance_callback
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)


**5. Capacity Constraint**

In [ ]:
# Import pandas to use isna()
import pandas as pd

# Define a callback function to return the demand for a given node.
def demand_callback(i):
    return demands[manager.IndexToNode(i)]


# Ensure all demands are valid numbers (If, replace NaN with 0).
for k in range(len(demands)):
    if pd.isna(demands[k]):
        # Use 0 to maintain float type if others are floats.
        demands[k] = 0


# Add the demand_callback with the routing model.
demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

"""
Add a capacity constraint to the routing model.
Total demand on any vehicle's route does not exceed its capacity.
Arguments:
 - demand_callback_index: The callback that provides the demand for each node.
 - 0 : Slack variable (allowing excess capacity).
 - vehicle_capacity * vehicle_count: Capacity of each vehicle.
 - True: Fix the start and end value to 0 (vehicles start and end empty).
 - Capacity : The name of this dimension.
"""

routing.AddDimensionWithVehicleCapacity(
    demand_callback_index, 0,
    [vehicle_capacity]*vehicle_count,
    True,
    "Capacity"
)

True

**6. Time-Duration Constraint**

In [ ]:
# Callback_function to calculate the time taken to travel between two nodes, including service time
def time_callback(i, j):
    # Convert solver internal indices to actual node indices
    from_node, to_node = manager.IndexToNode(i), manager.IndexToNode(j)

    # Calculate travel time: distance (m) / speed (m/min)
    # Assuming an average speed of 30 km/h, which is 500 meters per minute
    travel_time = distance_matrix[from_node][to_node] / 500

    # Add service time at the destination node.
    # No service time at the depot.
    service_time = SERVICE_TIME_MIN if from_node != 0 else 0

    # Return the total time (travel + service), cast to an integer
    return int(travel_time + service_time)


# Register the time_callback with the routing model
time_callback_index = routing.RegisterTransitCallback(time_callback)

"""
Add a time_dimension to the routing model.
This dimension tracks the accumulated time along each vehicle's route.
Arguments:
   - time_callback_index: callback that provides the time for traversing arcs.
   - 30: Slack var, depict the maximum wait time allowed at a node. (in mins).
   - 600: Maximum total time allowed for any single vehicle's route (in mins).
   - False: vehicles can start at different times.
   - Time : The name of this dimension.
"""

routing.AddDimension(
time_callback_index, 30, 400, False, "Time"
)

True

**7. Heuristic Approach: Rule of Thumb**

In [ ]:
# Configure the search parameters for the OR-Tools solver.
params = pywrapcp.DefaultRoutingSearchParameters()

"""
Solve Using First Solution Strategy, Clarke and Wright Savings Algorithm.
"""

# Generate an initial solution.
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.SAVINGS

# Shorter time limit for initial solution
params.time_limit.seconds = 30

print("Solving with SAVINGS heuristic, to get an initial solution...")
savings_solution = routing.SolveWithParameters(params)

# Print the initial Solution
print("Initial solution generated: ", savings_solution)

Solving with SAVINGS heuristic, to get an initial solution...
Initial solution generated:  Assignment(Capacity0 (0) | Capacity1 (920) | Capacity2 (4617) | Capacity3 (5717) | Capacity4 (1500) | Capacity5 (0) | Capacity6 (2497) | Capacity7 (2999) | Capacity8 (3189) | Capacity9 (0) | Capacity10 (2325) | Capacity11 (1523) | Capacity12 (4101) | Capacity13 (2700) | Capacity14 (2899) | Capacity15 (1130) | Capacity16 (1380) | Capacity17 (500) | Capacity18 (600) | Capacity19 (1910) | Capacity20 (2949) | Capacity21 (3340) | Capacity22 (460) | Capacity23 (3950) | Capacity24 (4740) | Capacity25 (0) | Capacity26 (5270) | Capacity27 (360) | Capacity28 (2350) | Capacity29 (1130) | Capacity30 (500) | Capacity31 (3025) | Capacity32 (3449) | Capacity33 (2937) | Capacity34 (5304) | Capacity35 (380) | Capacity36 (6041) | Capacity37 (4937) | Capacity38 (3660) | Capacity39 (1697) | Capacity40 (4641) | Capacity41 (3190) | Capacity42 (240) | Capacity43 (1796) | Capacity44 (6037) | Capacity45 (1237) | Capacity

**8. Metaheuristic Approach: Updating Initial Solution**

In [ ]:
"""
Solve Using GUIDED_LOCAL_SEARCH an Metaheuristics
"""

# Re-initialize params to ensure a clean state for the full search
params = pywrapcp.DefaultRoutingSearchParameters()

# Use SAVINGS for initial guess for GLS
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.SAVINGS
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH

# Original time limit for the full search
params.time_limit.seconds = 1800

print("\nSolving with GUIDED_LOCAL_SEARCH, (IMPROVING SAVINGS) for the final optimized solution...")
solution = routing.SolveWithParameters(params)

# Print the raw solution object returned by the OR-Tools solver.
print("Final solution generated: ", solution)

**5. Print the Solution Routes**

In [ ]:
# Import 'format_route.py' file

from src.format_routes import *

# Calling Function to print solution of route problem.
# Print the Initial Solution
print_route_details(savings_solution, manager, routing, vehicle_count, demands, "Initial Solution")
# Print the Final Solution
print_route_details(solution, manager, routing, vehicle_count, demands, "Final Solution")


--- Initial Solution ---
Objective: 6029156 meters
Route for vehicle 0:
 0 -> 979 -> 973 -> 934 -> 953 -> 986 -> 989 -> 924 -> 921 -> 910 -> 1015 -> 998 -> 960 -> 951 -> 928 -> 0
Distance of the route: 146035m
Load of the route: 6500

Route for vehicle 1:
 0 -> 898 -> 837 -> 894 -> 808 -> 831 -> 827 -> 821 -> 888 -> 790 -> 825 -> 844 -> 0
Distance of the route: 131540m
Load of the route: 6497

Route for vehicle 2:
 0 -> 759 -> 772 -> 854 -> 836 -> 901 -> 863 -> 773 -> 813 -> 779 -> 792 -> 891 -> 802 -> 782 -> 0
Distance of the route: 117979m
Load of the route: 6451

Route for vehicle 3:
 0 -> 835 -> 876 -> 880 -> 781 -> 858 -> 801 -> 887 -> 869 -> 770 -> 806 -> 0
Distance of the route: 115555m
Load of the route: 6397

Route for vehicle 4:
 0 -> 807 -> 830 -> 805 -> 763 -> 892 -> 875 -> 845 -> 775 -> 777 -> 829 -> 885 -> 793 -> 0
Distance of the route: 113351m
Load of the route: 6414

Route for vehicle 5:
 0 -> 855 -> 864 -> 500 -> 594 -> 615 -> 573 -> 536 -> 561 -> 600 -> 493 -> 565 -

**6. Visualizing the Routes Using Folium**

In [ ]:
# Import map_visualize from src
from src.map_visualize import *

In [ ]:
# Call 'map_visualize' function to generate appox. graph
generated_images = generate_vehicle_images(
    solution, manager, routing,
    locations, demands,
    vehicle_count,
    "Final Solution"
)

# Print the function
print("\nGenerated Images:")
for img in generated_images:
    print(img)


Generating maps for Final Solution...
✅ Vehicle 0 PNG saved
✅ Vehicle 1 PNG saved
✅ Vehicle 2 PNG saved
✅ Vehicle 3 PNG saved
✅ Vehicle 4 PNG saved
✅ Vehicle 5 PNG saved
✅ Vehicle 6 PNG saved
✅ Vehicle 7 PNG saved
✅ Vehicle 8 PNG saved
✅ Vehicle 9 PNG saved
✅ Vehicle 10 PNG saved
✅ Vehicle 11 PNG saved
✅ Vehicle 12 PNG saved
✅ Vehicle 13 PNG saved
✅ Vehicle 14 PNG saved
✅ Vehicle 15 PNG saved
✅ Vehicle 16 PNG saved
✅ Vehicle 17 PNG saved
✅ Vehicle 18 PNG saved
✅ Vehicle 19 PNG saved
✅ Vehicle 20 PNG saved
✅ Vehicle 21 PNG saved
✅ Vehicle 22 PNG saved
✅ Vehicle 23 PNG saved
✅ Vehicle 24 PNG saved
✅ Vehicle 25 PNG saved
✅ Vehicle 26 PNG saved
✅ Vehicle 27 PNG saved
✅ Vehicle 28 PNG saved
✅ Vehicle 29 PNG saved
✅ Vehicle 30 PNG saved
✅ Vehicle 31 PNG saved
✅ Vehicle 32 PNG saved
✅ Vehicle 33 PNG saved
✅ Vehicle 34 PNG saved
✅ Vehicle 35 PNG saved
✅ Vehicle 36 PNG saved
✅ Vehicle 37 PNG saved
✅ Vehicle 38 PNG saved
✅ Vehicle 39 PNG saved
✅ Vehicle 40 PNG saved
✅ Vehicle 41 PNG saved
✅ Veh

In [136]:
import os
import re # Import regex module for more flexible pattern matching

print("Deleting generated .png and .html files...")

# Get a list of all files in the current directory
current_files = os.listdir('.')

# Define patterns for the generated files
png_pattern = re.compile(r"^vehicle_(\d+)\.png$")
html_pattern = re.compile(r"^vehicle_(\d+)\.html$")
route_html_pattern = re.compile(r"^vehicle_route_(\d+)\.html$")

deleted_count = 0

for file_name in current_files:
    is_png = png_pattern.match(file_name)
    is_html = html_pattern.match(file_name)
    is_route_html = route_html_pattern.match(file_name)

    if is_png or is_html or is_route_html:
        try:
            os.remove(file_name)
            print(f"Deleted: {file_name}")
            deleted_count += 1
        except OSError as e:
            print(f"Error deleting {file_name}: {e}")

if deleted_count == 0:
    print("No generated .png or .html files found for deletion.")

print("\nCleanup complete.")

Deleting generated .png and .html files...
Deleted: vehicle_route_67.html
Deleted: vehicle_route_36.html
Deleted: vehicle_route_43.html
Deleted: vehicle_route_70.html
Deleted: vehicle_route_73.html
Deleted: vehicle_route_8.html
Deleted: vehicle_route_71.html
Deleted: vehicle_route_32.html
Deleted: vehicle_route_69.html
Deleted: vehicle_route_1.html
Deleted: vehicle_route_51.html
Deleted: vehicle_route_17.html
Deleted: vehicle_route_18.html
Deleted: vehicle_route_63.html
Deleted: vehicle_route_37.html
Deleted: vehicle_route_49.html
Deleted: vehicle_route_42.html
Deleted: vehicle_route_58.html
Deleted: vehicle_route_80.html
Deleted: vehicle_route_35.html
Deleted: vehicle_route_31.html
Deleted: vehicle_route_68.html
Deleted: vehicle_route_74.html
Deleted: vehicle_route_46.html
Deleted: vehicle_route_76.html
Deleted: vehicle_route_4.html
Deleted: vehicle_route_28.html
Deleted: vehicle_route_39.html
Deleted: vehicle_route_24.html
Deleted: vehicle_route_27.html
Deleted: vehicle_route_20.html

In [26]:
def get_osrm_route(coords):
    if len(coords) < 2:
        return None, 0, 0

    coord_str = ";".join([f"{lon},{lat}" for lat, lon in coords])

    url = f"http://router.project-osrm.org/route/v1/driving/{coord_str}?overview=full&geometries=geojson"

    try:
        res = requests.get(url, timeout=10).json()
        if "routes" in res and len(res["routes"]) > 0:
            route = res["routes"][0]
            return route["geometry"], route["distance"], route["duration"]
        else:
            return None, 0, 0
    except Exception as e:
        print(f"Error fetching OSRM route: {e}")
        return None, 0, 0

In [27]:
# To get the summary result table...
import pandas as pd

def generate_route_summary_table(solution, manager, routing, locations, demands, vehicle_count):
    route_summaries = []

    print("Collecting route data for table...")

    for vehicle_id in range(vehicle_count):
        index = routing.Start(vehicle_id)
        route_nodes_indices = []
        route_demands = 0
        num_stops = 0
        current_route_coords = []

        # Skip unused vehicles
        if solution.Value(routing.NextVar(index)) == routing.End(vehicle_id):
            continue

        # Add depot to route coordinates
        depot_node = manager.IndexToNode(routing.Start(vehicle_id))
        current_route_coords.append(locations[depot_node])

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            current_route_coords.append(locations[node])

            if node != 0:  # Exclude depot from stops count and demand
                num_stops += 1
                route_demands += demands[node]

            index = solution.Value(routing.NextVar(index))

        # Add final depot to route coordinates
        final_depot_node = manager.IndexToNode(routing.End(vehicle_id))
        current_route_coords.append(locations[final_depot_node])

        # Get OSRM estimated distance and duration
        # Ensure get_osrm_route returns 3 values: geojson, distance, duration
        geojson, osrm_distance, osrm_duration = get_osrm_route(current_route_coords)

        # Convert OSRM duration from seconds to minutes
        osrm_duration_min = osrm_duration / 60

        route_summaries.append({
            "Vehicle ID": vehicle_id,
            "No. of Stops": num_stops,
            "Total Daily Demand (kg)": route_demands,
            "Est. Distance (km)": round(osrm_distance / 1000, 2),
            "Est. Time (min)": round(osrm_duration_min, 2)
        })

    df_summary = pd.DataFrame(route_summaries)
    return df_summary

# Generate and print the summary table for the Final Solution
summary_df = generate_route_summary_table(solution, manager, routing, locations, demands, vehicle_count)
print("\n--- Route Summary Table (Final Solution) ---")
# Convert DataFrame into the CSV file
summary_df.to_csv('route_result_table.csv', index=False)
summary_df


--- Route Summary Table (Final Solution) ---


,Vehicle ID,No. of Stops,Total Daily Demand (kg),Est. Distance (km),Est. Time (min)
0,0,14,6500,221.30,244.41
1,1,11,6497,170.80,210.45
2,2,13,6451,155.14,181.32
3,3,10,6397,150.37,173.43
4,4,12,6414,150.92,175.04
...,...,...,...,...,...
79,79,11,5857,56.34,81.92
80,80,12,5905,44.00,66.24
81,81,11,6377,37.25,59.54
82,82,11,6190,31.07,50.23


**Total Estimated Distance get through OSRM**

In [30]:
# Total Estimated distance by each vehicle.
print('Total Esti. Distance by 84 Vehicle: ', summary_df['Est. Distance (km)'].sum())

Total Esti. Distance by 84 Vehicle:  9031.529999999999


**Total Distance covered by all 87 vehicles before optimization**

In [14]:
# Load the JCCI Dataset
import pandas as pd
df_ = pd.read_excel('jcci_vehicle_data_dist_2_march_apr.xlsx')
df_.head()

# Removing the column vehicle_ID and remain only Day_[] columns
total_distance = df_.iloc[:, 1:].values.sum()

# # average daily distance
before = total_distance / 60

# Before Optimization Distance is
print("Before Distance:", before)

Before Distance: 10213.485000000002
